# シーン検索デモを Colab で起動する（A100）

2 段構成（VideoJev-4B → 学習済み Qwen3-VL-4B）+ 3 段の振り分け。起動すると共有リンク（gradio.live）が出るので、ブラウザで開く。

前提: Drive の `MyDrive/movie_llm/grpo3_time/step_00400/`（LoRA）と、`videos_2fps_s360.zip`（サンプルに使う縮小済み動画）。

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "transformers==5.14.1" "accelerate>=1.10" "peft>=0.15" "einops>=0.8" "scipy>=1.12" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "gradio>=5"
!pip install -q "flash-linear-attention==0.4.2" "fla-core==0.4.2" || echo 'fla なしで続行'
!pip uninstall -y -q torchao
!pip install -q -e . --no-deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
import os, shutil, zipfile
os.makedirs('/content/samples', exist_ok=True)
# サンプル: 縮小済みのベンチマーク動画から 3 本（自分の素材があれば /content/samples に置く）
with zipfile.ZipFile(f'{DRIVE}/videos_2fps_s360.zip') as z:
    for name in ['DdzGeAGSpok.mp4', 'XOiSYSLoKSc.mp4', '_VpWJ9DxTFg.mp4']:
        if name in z.namelist():
            z.extract(name, '/content/samples')
print('samples', os.listdir('/content/samples'))

In [ ]:
%cd /content/movie_llm
import os
os.environ['MOVIE_LLM_SAMPLES'] = '/content/samples'
!python -m movie_llm.demo.app --adapter {DRIVE}/grpo3_time/step_00400 --max-frames 2048 --share

出力の `Running on public URL: https://xxxx.gradio.live` を開く。初回はモデルの読み込みで 1〜2 分、1 回の検索は 10〜20 秒。